In [7]:
# -------------------------------------------------------
# 05_neo4j_rag_demo_st.ipynb
# Neo4j Graph-RAG Demo (SentenceTransformers version)
# -------------------------------------------------------

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from pathlib import Path
from dotenv import load_dotenv
import os

# -------------------------------------------------------
# Load environment
# -------------------------------------------------------
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# -------------------------------------------------------
# Paths
# -------------------------------------------------------
BASE_DIR = Path("graph")
IDX_DIR = BASE_DIR / "indices"
EMB_DIR = BASE_DIR / "embeddings"

# -------------------------------------------------------
# Load metadata + FAISS index
# -------------------------------------------------------
meta = pd.read_parquet(EMB_DIR / "metadata.parquet")

index = faiss.read_index(
    str(IDX_DIR / "faiss_index_st.index")
)

print("FAISS vectors:", index.ntotal)
print("Metadata records:", len(meta))

# -------------------------------------------------------
# Load SentenceTransformer
# -------------------------------------------------------
MODEL_NAME = "all-MiniLM-L6-v2"
st_model = SentenceTransformer(MODEL_NAME)

# -------------------------------------------------------
# Retrieve from Graph chunks
# -------------------------------------------------------
def retrieve(query: str, k: int = 5):
    query_vec = st_model.encode([query], convert_to_numpy=True)
    query_vec = query_vec.astype("float32")

    distances, indices = index.search(query_vec, k)

    return [meta.iloc[i]["text"] for i in indices[0]]

# -------------------------------------------------------
# Graph-RAG Answer (Sinhala)
# -------------------------------------------------------
def rag_answer(query: str, k: int = 5) -> str:
    context_chunks = retrieve(query, k)
    context = "\n".join(context_chunks)

    prompt = f"""
ඔබ ආයුර්වේද වෛද්‍ය සහායකයෙකි.

පහත Neo4j graph මගින් ලබාගත් සන්දර්භය (Context) භාවිතා කර
ප්‍රශ්නයට **සිංහල භාෂාවෙන්** පිළිතුරු දෙන්න.

Context:
{context}

Question:
{query}

Instructions:
- සිංහලෙන් පිළිතුරු දෙන්න
- වෛද්‍යමය ලෙස නිවැරදි විය යුතුය
- ලබාදුන් සන්දර්භය අනුව රෝගයේ නාමය පැහැදිලිව සඳහන් කරන්න

Answer (සිංහලෙන්):
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    return response.choices[0].message.content.strip()

# -------------------------------------------------------
# Demo
# -------------------------------------------------------
query = "බෙල්ලේ තද බවක් දැනීමට, බඩ පුරවා දැමීම, නිදා ගැනීමේ අපහසුතාව කුමන රෝගයක් නිසා ඇති විය හැකිද?"
print(rag_answer(query))

FAISS vectors: 48
Metadata records: 48
බෙල්ලේ තද බවක් දැනීම, බඩ පුරවා දැමීම සහ නිදා ගැනීමේ අපහසුතාවය යන ලක්ෂණ, සාමාන්‍යයෙන් ආහාර පෝෂණය හා සම්බන්ධ ගැටළු, ආන්තික ආසාදන, හෝ මනෝභාවය හා සම්බන්ධ ගැටළු මගින් ඇති විය හැක. 

මෙම ලක්ෂණ තුනම "ආහාර පෝෂණය හා සම්බන්ධ රෝග" හෝ "මනෝභාවය හා සම්බන්ධ රෝග" යන කාණ්ඩයට අයත් වේ. 

ඒ අනුව, මෙම ලක්ෂණ ඇති විය හැකි රෝගයක් ලෙස "ආහාර පෝෂණය හා සම්බන්ධ ගැටළු" හෝ "මනෝභාවය හා සම්බන්ධ ගැටළු" යනුවෙන් හැඳින්විය හැක. 

නමුත්, සම්පූර්ණ නිවැරදි වෛද්‍ය ප්‍රතිකාරයක් ලබා ගැනීමට, වෛද්‍යවරයෙකුගේ උපදෙස් ලබා ගැනීම අත්‍යවශ්‍ය වේ.
